# Analytics Hub - Support file

Stuck? This notebook makes a **support file** you can email to us, for setups made by hand. If you used the
installer, run `valuelens-install --support-bundle` instead (or use *Having trouble? Create a support file* in its menu).

It only reads, and writes one zip to `Files/support/` in the default Lakehouse:

| File | What it holds |
|---|---|
| `README.txt`, `manifest.json` | What's inside and what was removed |
| `settings.json` | The workspace, Lakehouse and capacity IDs |
| `runs.json` | Recent pipeline, notebook and semantic model runs with their errors, and the latest `load_log` statuses |
| `row-counts.json` | Rows per table, so an empty table shows at once. Counts only, never the rows |
| `logs-summary.json` | Counts of 429, 401 and 403 errors, time-outs and windows, and the first error |
| `last-run.json`, `prereqs.json` | Not collected here: the installer keeps these |

Secrets, tokens and keys are removed. Email addresses, UPNs and names are replaced by `user-` and 10 hex digits
(SHA-256 with a random salt that's kept nowhere). The notebook prints every file, so you can check it before you send it.

**To use it:** attach the Analytics Hub Lakehouse as the default Lakehouse, then **Run all**.

In [ ]:
# Settings. The defaults suit most installs.
OUT_DIR = 'Files/support'   # Where the zip goes, in the default Lakehouse.
JOBS_PER_ITEM = 5           # Recent runs kept for each pipeline, notebook and semantic model.
MAX_ITEMS = 60              # The most notebooks and models read.
MAX_TABLES = 300            # The most tables counted.
LOAD_LOG_ROWS = 60          # The latest load_log rows kept (status and reason only).

In [ ]:
# Redaction and the file layout. These match the installer's --support-bundle exactly.
import hashlib
import io
import json
import os
import re
import zipfile
from datetime import datetime, timezone

SUPPORT_PRODUCT = 'Analytics Hub'
SUPPORT_PREFIX = 'analyticshub-support'
SECRET_KEY = re.compile(r'secret|passw(or)?d|^pwd$|token|key(?!vault)|signature|connectionstring|^sas|sas(url|uri)?$|credential|authori[sz]ation|cookie', re.I)
PERSON_KEY = re.compile(r'^(givenName|surname|fullName|userName|contact|owner|owners|createdBy|lastModifiedBy|requestedBy|upn|userPrincipalName|mail|email|adminEmail|user|signedInAs)$', re.I)
GUID = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.I)
ISO_DATE = re.compile(r'^\d{4}-\d{2}-\d{2}([T ][\d:.]+(Z|[+-]\d{2}:?\d{2})?)?$')
EMAIL = re.compile(r"[A-Za-z0-9._%+'#-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}")
SECRET_TEXT = [
    (re.compile(r'eyJ[A-Za-z0-9_-]{8,}\.[A-Za-z0-9_-]{8,}\.[A-Za-z0-9_-]*'), '[removed]'),
    (re.compile(r'\bBearer\s+[A-Za-z0-9._~+/=-]{8,}', re.I), 'Bearer [removed]'),
    (re.compile(r'''\b(sig|AccountKey|SharedAccessKey|SharedAccessSignature|client_secret|client_assertion|code_verifier|refresh_token|access_token|id_token|password|pwd)(\s*[=:]\s*)("?)[^;&\s"',}]+''', re.I), r'\1\2\3[removed]'),
    (re.compile(r'\b[A-Za-z0-9_.-]{2,5}~[A-Za-z0-9_.~-]{30,}'), '[removed]'),
]
LONG_BASE64 = re.compile(r'[A-Za-z0-9+/_=-]{80,}')
RESOURCE_PATH = re.compile(r'/(subscriptions|resourceGroups|providers|groups|workspaces|items)/', re.I)
ANSI = re.compile(r'\x1b\[[0-9;]*m')
REDACTION_RULES = [
    'Values under keys that name a secret, token, key, password, signature, SAS, credential or connection string are removed. GUIDs, dates, numbers and true/false under them stay, such as when a secret expires.',
    'Tokens (JWTs), Bearer headers, SAS signatures, account and shared access keys, client secrets and long base64 strings are removed from any text.',
    "Email addresses, UPNs and people's names are replaced by user- and 10 hex digits: a SHA-256 hash with a random salt that is used for this file only and kept nowhere. The same person has the same token throughout the file.",
    'Tenant, workspace and item IDs (GUIDs) are kept, so we can find the right run.',
    'No data rows are read: tables appear as row counts only.',
]
LOG_PATTERNS = {
    'http429': re.compile(r'\b429\b|too many requests|throttl', re.I),
    'http401': re.compile(r'\b401\b|unauthori[sz]ed|invalid_client|AADSTS7000215|AADSTS7000222|expired (client )?secret', re.I),
    'http403': re.compile(r'\b403\b|forbidden|insufficient privileges|AADSTS65001|not authori[sz]ed', re.I),
    'timeouts': re.compile(r'timed? ?out|timeout|deadline exceeded|ETIMEDOUT|ESOCKETTIMEDOUT', re.I),
    'windows': re.compile(r'\bwindows?\s+\d+(\s*(of|/)\s*\d+)?', re.I),
}
ERROR_LINE = re.compile(r'error|failed|exception|denied|refused', re.I)


class Redactor:
    """Removes secrets and hashes email addresses, UPNs and names: SHA-256 of a salt and the
    lower-cased value, shown as user- and 10 hex digits. The salt is random for each file and kept nowhere."""

    def __init__(self, salt=None):
        self.salt = salt if salt is not None else os.urandom(16)
        self.counts = {'secrets': 0, 'emails': 0, 'names': 0}
        self.names = set()

    def hash(self, value):
        return 'user-' + hashlib.sha256(self.salt + value.strip().lower().encode('utf-8')).hexdigest()[:10]

    def _secret(self, to):
        def sub(m):
            self.counts['secrets'] += 1
            return m.expand(to)
        return sub

    def text(self, s):
        out = ANSI.sub('', s)
        for pattern, to in SECRET_TEXT:
            out = pattern.sub(self._secret(to), out)

        def base64(m):
            v = m.group(0)
            if RESOURCE_PATH.search(v) or not (re.search(r'\d', v) and re.search(r'[A-Z]', v) and re.search(r'[a-z]', v)):
                return v
            self.counts['secrets'] += 1
            return '[removed]'
        out = LONG_BASE64.sub(base64, out)

        def email(m):
            self.counts['emails'] += 1
            return self.hash(m.group(0))
        out = EMAIL.sub(email, out)
        for name in sorted(self.names, key=len, reverse=True):
            def person(m, name=name):
                self.counts['names'] += 1
                return self.hash(name)
            out = re.sub(r'(?<![\w-])' + re.escape(name) + r'(?![\w-])', person, out, flags=re.I)
        return out

    def _walk(self, v, secret, person):
        if v is None or isinstance(v, (bool, int, float)):
            return v
        if isinstance(v, str):
            if secret:
                if GUID.match(v) or ISO_DATE.match(v):
                    return v
                self.counts['secrets'] += 1
                return '[removed]'
            if person and v.strip() and not GUID.match(v):
                self.counts['emails' if '@' in v else 'names'] += 1
                return self.hash(v)
            return self.text(v)
        if isinstance(v, (list, tuple)):
            return [self._walk(x, secret, person) for x in v]
        if isinstance(v, dict):
            return {k: self._walk(x, secret or bool(SECRET_KEY.search(k)), person or bool(PERSON_KEY.search(k))) for k, x in v.items()}
        return self.text(str(v))

    def learn(self, v, person=False):
        """Notes the names under person keys, so the same names in free text are hashed too."""
        if isinstance(v, str):
            if person and len(v.strip()) >= 3 and '@' not in v and not GUID.match(v):
                self.names.add(v.strip())
        elif isinstance(v, (list, tuple)):
            for x in v:
                self.learn(x, person)
        elif isinstance(v, dict):
            for k, x in v.items():
                self.learn(x, person or bool(PERSON_KEY.search(k)))

    def value(self, v):
        return self._walk(v, False, False)


def not_collected(why):
    return {'notCollected': why}


def summarise_log(text):
    """Counts throttling, sign-in failures, time-outs and windows in some text, and keeps its first error."""
    out = {k: len(p.findall(text)) for k, p in LOG_PATTERNS.items()}
    lines = text.splitlines() if text else []
    out['lines'] = len(lines)
    out['firstError'] = next((l.strip()[:600] for l in lines if ERROR_LINE.search(l)), None)
    return out


def support_file_name(d):
    return f"{SUPPORT_PREFIX}-{d.astimezone(timezone.utc).strftime('%Y%m%d-%H%M%S')}.zip"


def build_support_files(s, salt=None):
    """The redacted files of a support file, in the installer's order: (name, text) pairs."""
    r = Redactor(salt)
    r.learn(s)

    def as_json(v):
        return json.dumps(r.value(v), indent=2, ensure_ascii=False) + '\n'
    body = [(name, as_json(s[key])) for name, key in [
        ('settings.json', 'settings'), ('last-run.json', 'lastRun'), ('prereqs.json', 'prereqs'),
        ('runs.json', 'runs'), ('row-counts.json', 'rowCounts'), ('logs-summary.json', 'logsSummary')]]
    who = r.value({'upn': (s.get('user') or {}).get('upn')})['upn'] or 'unknown'
    readme = '\n'.join([
        f'{SUPPORT_PRODUCT} support file',
        '',
        f"Made by {s['madeBy']} on {s['createdAt'][:19].replace('T', ' ')} UTC, for the {'Azure' if s['route'] == 'azure' else 'Fabric'} route.",
        f'Run as {who}, so it includes run statuses and row counts that person can see.' if s['signedIn'] else 'Made without signing in, so run statuses and row counts say "notCollected".',
        '',
        'Files:',
        '  manifest.json       What is in this file and what redaction removed.',
        '  settings.json       The install record: IDs and choices.',
        '  last-run.json       The last installer run, step by step, with the step that failed and its error.',
        '  prereqs.json        Roles, licences, capacities and permissions: which are met and which are missing.',
        '  runs.json           Recent pipeline, notebook, refresh and Azure job runs, with their errors.',
        '  row-counts.json     Rows per table. Counts only, never the rows.',
        '  logs-summary.json   Counts of throttling (429), sign-in (401) and permission (403) errors, time-outs and windows.',
        '',
        'What was removed or hidden:',
        *[f'  - {rule}' for rule in REDACTION_RULES],
        '',
        'Every file is plain text. Open them to check before you send this file.',
        '',
    ])
    files = [('README.txt', readme)] + body
    manifest = {
        'product': SUPPORT_PRODUCT,
        'installerVersion': s.get('installerVersion'),
        'madeBy': s['madeBy'],
        'createdAt': s['createdAt'],
        'route': s['route'],
        'signedIn': s['signedIn'],
        'files': [{'name': n, 'bytes': len(t.encode('utf-8'))} for n, t in files],
        'redaction': {'rules': REDACTION_RULES, 'counts': dict(r.counts)},
    }
    return [files[0], ('manifest.json', json.dumps(manifest, indent=2, ensure_ascii=False) + '\n')] + body


def support_zip(files):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as z:
        for name, text in files:
            z.writestr(name, text)
    return buf.getvalue()

In [ ]:
# Collect, redact and write the support file. It only reads: nothing in the workspace changes.
import requests

now = datetime.now(timezone.utc)
ctx = notebookutils.runtime.context
workspace_id = ctx.get('currentWorkspaceId')
token = notebookutils.credentials.getToken('pbi')
headers = {'Authorization': f'Bearer {token}'}


def get(url):
    r = requests.get(url, headers=headers, timeout=60)
    if r.status_code >= 400:
        raise RuntimeError(f'HTTP {r.status_code}: {r.text[:300]}')
    return r.json()


def attempt(fn):
    try:
        return fn()
    except Exception as e:  # Each part says why it's missing rather than stopping the file.
        return not_collected(str(e)[:500])


FABRIC = f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}'


def items(kind):
    return get(f'{FABRIC}/items?type={kind}').get('value', [])


def jobs(item, top):
    runs = get(f"{FABRIC}/items/{item['id']}/jobs/instances").get('value', [])
    runs.sort(key=lambda j: j.get('startTimeUtc') or '', reverse=True)
    return {'name': item.get('displayName'), 'id': item['id'], 'jobs': [{
        'id': j.get('id'), 'jobType': j.get('jobType'), 'invokeType': j.get('invokeType'), 'status': j.get('status'),
        'startTimeUtc': j.get('startTimeUtc'), 'endTimeUtc': j.get('endTimeUtc'),
        'error': (j.get('failureReason') or {}).get('message'), 'errorCode': (j.get('failureReason') or {}).get('errorCode'),
    } for j in runs[:top]]}


def refreshes(model):
    rows = get(f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{model['id']}/refreshes?$top={JOBS_PER_ITEM}").get('value', [])
    return {'name': model.get('displayName'), 'id': model['id'], 'refreshes': [{
        'status': x.get('status'), 'refreshType': x.get('refreshType'), 'startTime': x.get('startTime'), 'endTime': x.get('endTime'),
        'error': x.get('serviceExceptionJson'),
    } for x in rows]}


def load_log():
    cols = 'run_id, activity, source, status, reason_kind, reason, attempts, started_at, ended_at, error_code, error_message'
    df = spark.sql(f'SELECT {cols} FROM load_log ORDER BY logged_at DESC LIMIT {int(LOAD_LOG_ROWS)}')
    return [{k: (v.isoformat() if hasattr(v, 'isoformat') else v) for k, v in row.asDict().items()} for row in df.collect()]


def row_counts():
    tables = {}
    for db in spark.catalog.listDatabases():
        for t in spark.catalog.listTables(db.name):
            if t.tableType == 'VIEW' or len(tables) >= MAX_TABLES:
                continue
            name = f'{db.name}.{t.name}'
            try:
                tables[name] = {'rows': spark.table(f'`{db.name}`.`{t.name}`').count()}
            except Exception as e:
                tables[name] = not_collected(str(e)[:300])
    out = {'tables': tables, 'empty': sorted(n for n, v in tables.items() if v.get('rows') == 0)}
    path = '/lakehouse/default/Files/valuelens_installer/data_check.json'
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            out['dataCheck'] = json.load(f)
    return out


def fabric_runs():
    out = {}
    out['pipelines'] = attempt(lambda: [jobs(p, JOBS_PER_ITEM) for p in items('DataPipeline')])
    out['notebooks'] = attempt(lambda: [jobs(n, 3) for n in items('Notebook')[:MAX_ITEMS]])
    out['semanticModels'] = attempt(lambda: [refreshes(m) for m in items('SemanticModel')[:MAX_ITEMS]])
    out['loadLog'] = attempt(load_log)
    return out


def error_texts(v, into):
    if isinstance(v, list):
        for x in v:
            error_texts(x, into)
    elif isinstance(v, dict):
        for k, x in v.items():
            if k in ('error', 'error_message', 'reason') and x:
                into.append(x if isinstance(x, str) else json.dumps(x))
            else:
                error_texts(x, into)


runs = attempt(fabric_runs)
errors = []
error_texts(runs, errors)
data = {
    'madeBy': 'the AnalyticsHub_Support_File notebook',
    'installerVersion': None,
    'createdAt': now.isoformat().replace('+00:00', 'Z'),
    'route': 'fabric',
    'signedIn': True,
    'user': {'upn': ctx.get('userName')},
    'settings': {
        'note': 'Made by hand: there is no install record. These are the notebook\'s workspace and Lakehouse.',
        'tenantId': ctx.get('tenantId') or ctx.get('currentTenantId'),
        'workspace': {'id': workspace_id, 'name': ctx.get('currentWorkspaceName')},
        'lakehouse': {'id': ctx.get('defaultLakehouseId'), 'name': ctx.get('defaultLakehouseName')},
        'capacityId': ctx.get('capacityId'),
        'sparkRuntime': ctx.get('runtimeVersion') or spark.version,
    },
    'lastRun': not_collected('Made by hand: the installer keeps the run log. To check the setup, run the installer\'s prereqs command.'),
    'prereqs': not_collected('Made by hand. The installer\'s prereqs command checks roles, licences and permissions.'),
    'runs': runs,
    'rowCounts': attempt(row_counts),
    'logsSummary': {
        'installerRuns': not_collected('Made by hand: there are no installer runs.'),
        'runErrors': summarise_log('\n'.join(errors)),
    },
}

files = build_support_files(data)
name = support_file_name(now)
folder = f"/lakehouse/default/{OUT_DIR.strip('/')}"
os.makedirs(folder, exist_ok=True)
with open(f'{folder}/{name}', 'wb') as f:
    f.write(support_zip(files))

print(f'Support file: {OUT_DIR}/{name}')
for n, t in files:
    print(f"  {n:<20} {len(t.encode('utf-8')):>8,} bytes")
print('\nSecrets are removed and people\'s names and email addresses are hashed. Here is everything in it:\n')
for n, t in files:
    print(f'===== {n} =====\n{t}')
print(f'Download it from the Lakehouse ({OUT_DIR}), then email it to us.')